# Probe — {{NAME}} against {{BASELINE}}

**A screening run, not the benchmark.** ResNet-18 and a stratified slice, sized so
several seeds finish in the time one full run would take. A reduced setting can
invert a result: a method that needs capacity or volume to show its advantage
loses here and wins at full scale.

Every number below is printed with the reduction that produced it. Read them
together — a number that can be read without its reduction gets misquoted.

Proposal revision under test: `{{REVISION}}`.


## The common environment

Both implementations see exactly this: same dataset, same slice, same seeds. The
slice is drawn per class, not at random, because the proposal requires every class
to be present in the source collection the local correspondence uses — a random
slice can drop one and leave that correspondence undefined, which would look like
a defect of the method and be a defect of the sampling.


In [ ]:
import json, pathlib, subprocess, sys

ROOT = pathlib.Path.cwd().parents[1]  # <repo>/{{NAME}}/Notebooks -> <repo>
HERE = pathlib.Path.cwd()

REDUCTION = {
    "dataset": "{{DATASET}}",
    "backbone": "resnet18",
    "fraction": {{FRACTION}},
    "epochs": {{EPOCHS}},
    "batch_size": 64,
    "seeds": {{SEEDS}},
    "revision": "{{REVISION}}",
    "device": "cpu",
}

config = HERE / "probe_config.json"
config.write_text(json.dumps(REDUCTION, indent=2), encoding='utf-8')
REDUCTION


## Train and measure

`benchmark.py` trains each implementation over every seed and measures accuracy,
wall time, peak memory and parameter count. An implementation that cannot be
driven into this exact reduction as it stands is reported as `not applicable`
with its reason — the baseline is the user's prior work and is never edited to
make a comparison possible.


In [ ]:
OUT = ROOT / "{{NAME}}" / "Results" / "{{PROBE_RESULTS}}"

completed = subprocess.run(
    [sys.executable, str(HERE / "benchmark.py"),
     "--config", str(config), "--out", str(OUT), "--data", str(ROOT / ".benchmark-data")],
    cwd=str(HERE), text=True, capture_output=True)
print(completed.stdout[-4000:] or completed.stderr[-4000:])
completed.check_returncode()


## What it found

Cost compares cleanly even when the two predict on different statistical units.
Accuracy does not: if one predicts per instance and the other per bag, a single
number would require inventing an aggregation rule that can dominate what it
claims to measure. **`not applicable` is a legitimate cell.**


In [ ]:
summary = json.loads(OUT.read_text(encoding='utf-8'))

print(f"screening under {summary['reduction']['dataset']}, resnet18, fraction={summary['reduction']['fraction']}, "
      f"seeds={len(summary['reduction']['seeds'])}, revision={summary['revision']}")
print()
print(f"{'':<12}{'accuracy':>22}{'seconds':>18}{'peak MiB':>14}{'params':>12}")
for label, row in summary['comparison'].items():
    if not row.get('applicable'):
        print(f"{label:<12}{'not applicable':>22}   {row['reason'][:60]}")
        continue
    acc, sec, mem = row['accuracy'], row['seconds'], row['peakMiB']
    print(f"{label:<12}"
          f"{acc['mean']:.4f} ± {acc['stdev']:.4f}".rjust(22) +
          f"{sec['mean']:.1f} ± {sec['stdev']:.1f}".rjust(18) +
          f"{mem['mean']:.1f}".rjust(14) +
          f"{row['parameters']:,}".rjust(12))
